acuuracy = correct / total

A prediction counts as correct only if:
- the model answer
- exactly matches (case-insensitive)
- the majority human annotation for that question


In [3]:
import json
import pandas as pd

In [5]:

def compute_strict_accuracy(
    json_path: str,
    csv_path: str,
    id_col: str = None,
    pred_col: str = None,
):
    # -----------------------------
    # Load gold annotations (JSON)
    # -----------------------------
    with open(json_path, "r", encoding="utf-8") as f:
        gold = json.load(f)

    # Build majority-vote gold answers
    gold_majority = {}
    skipped = 0

    for qid, qdata in gold.items():
        annotations = qdata.get("annotations", [])
        if not annotations:
            skipped += 1
            continue

        majority = max(annotations, key=lambda x: x["count"])
        answers = majority.get("en_answers") or majority.get("answers")

        if not answers:
            skipped += 1
            continue

        gold_majority[qid] = answers[0].strip().lower()

    # -----------------------------
    # Load model predictions (CSV)
    # -----------------------------
    pred = pd.read_csv(csv_path)

    # Auto-detect columns if not given
    if id_col is None:
        id_col = pred.columns[0]

    if pred_col is None:
        # common names
        candidates = ["prediction", "answer", "response", "model_answer"]
        found = [c for c in pred.columns if c.lower() in candidates]
        pred_col = found[0] if found else pred.columns[-1]

    # -----------------------------
    # Compute strict accuracy
    # -----------------------------
    correct = 0
    total = 0

    for _, row in pred.iterrows():
        qid = row[id_col]
        if qid not in gold_majority:
            continue

        model_answer = str(row[pred_col]).strip().lower()
        gold_answer = gold_majority[qid]

        if model_answer == gold_answer:
            correct += 1
        total += 1

    accuracy = correct / total if total > 0 else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "skipped": skipped,
    }


In [14]:
result = compute_strict_accuracy(
    json_path = "data/annotations/China_data.json",
    csv_path  = "model_inference_results/Qwen2.5-3B-Instruct-China_Chinese_inst-4_result.csv"
)

print("""
• The model produced predictions for 56 questions
• 6 could not be evaluated (no valid gold label)
• 50 were evaluated
• 3 exactly matched the majority human answer
• Final strict accuracy = 3 / 50 = 6%""")
print(result)


• The model produced predictions for 56 questions
• 6 could not be evaluated (no valid gold label)
• 50 were evaluated
• 3 exactly matched the majority human answer
• Final strict accuracy = 3 / 50 = 6%
{'accuracy': 0.06, 'correct': 3, 'total': 50, 'skipped': 6}


In [7]:
result = compute_strict_accuracy(
    json_path = "data/annotations/Mexico_data.json",
    csv_path  = "model_inference_results/mt5-small-Mexico_Spanish_inst-4_result.csv"
)
print(result)

{'accuracy': 0.0, 'correct': 0, 'total': 98, 'skipped': 10}


In [8]:
result = compute_strict_accuracy(
    json_path = "data/annotations/US_data.json",
    csv_path  = "model_inference_results/mt5-small-US_English_inst-4_result.csv"
)
print(result)

{'accuracy': 0.0, 'correct': 0, 'total': 5, 'skipped': 5}


In [9]:
result = compute_strict_accuracy(
    json_path = "data/annotations/China_data.json",
    csv_path  = "model_inference_results/Qwen2.5-3B-Instruct-China_Chinese_inst-4_result.csv"
)
print(result)

{'accuracy': 0.06, 'correct': 3, 'total': 50, 'skipped': 6}


In [10]:
result = compute_strict_accuracy(
    json_path = "data/annotations/Mexico_data.json",
    csv_path  = "model_inference_results/mt5-small-Mexico_Spanish_inst-4_result.csv"
)
print(result)

{'accuracy': 0.0, 'correct': 0, 'total': 98, 'skipped': 10}


In [11]:
result = compute_strict_accuracy(
    json_path = "data/annotations/US_data.json",
    csv_path  = "model_inference_results/Qwen2.5-3B-Instruct-US_English_inst-4_result.csv"
)
print(result)

{'accuracy': 0.3, 'correct': 21, 'total': 70, 'skipped': 5}


In [13]:
result = compute_strict_accuracy(
    json_path = "data/annotations/US_data.json",
    csv_path  = "model_inference_results/t5-small-US_English_inst-4_result.csv"
)
print(result)

{'accuracy': 0.01, 'correct': 1, 'total': 100, 'skipped': 5}
